In [1]:
import numpy as np
import pandas as pd

def clean_openface_df(df, confidence_threshold=0.6):
    df = df.copy()

    df = df.replace([np.inf, -np.inf], np.nan)

    if "success" in df.columns:
        df = df[df["success"] == 1]

    if "confidence" in df.columns:
        df = df[df["confidence"] >= confidence_threshold]

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].interpolate().bfill().ffill()

    return df

In [2]:
def aggregate_columns(df, cols, prefix):
    features = {}

    existing_cols = [c for c in cols if c in df.columns]

    for col in existing_cols:
        x = df[col].dropna().values

        if len(x) == 0:
            continue

        features[f"{prefix}_{col}_mean"] = np.mean(x)
        features[f"{prefix}_{col}_std"] = np.std(x)
        features[f"{prefix}_{col}_min"] = np.min(x)
        features[f"{prefix}_{col}_max"] = np.max(x)
        features[f"{prefix}_{col}_median"] = np.median(x)
        features[f"{prefix}_{col}_p25"] = np.percentile(x, 25)
        features[f"{prefix}_{col}_p75"] = np.percentile(x, 75)

    return features

In [3]:
def extract_derived_video_features(df):
    features = {}

    # Smile intensity
    if "AU06_r" in df.columns and "AU12_r" in df.columns:
        smile = df["AU06_r"] + df["AU12_r"]
        features["derived_smile_mean"] = smile.mean()
        features["derived_smile_std"] = smile.std()
        features["derived_smile_rate"] = (smile > smile.quantile(0.75)).mean()

    # Negative/tension expression
    if "AU04_r" in df.columns and "AU15_r" in df.columns:
        neg = df["AU04_r"] + df["AU15_r"]
        features["derived_negative_au_mean"] = neg.mean()
        features["derived_negative_au_std"] = neg.std()

    # Overall AU expressiveness
    au_r_cols = [c for c in df.columns if c.startswith("AU") and c.endswith("_r")]
    if len(au_r_cols) > 0:
        au_activity = df[au_r_cols].mean(axis=1)
        features["derived_au_expressiveness_mean"] = au_activity.mean()
        features["derived_au_expressiveness_std"] = au_activity.std()

    # Blink activity
    if "AU45_c" in df.columns:
        features["derived_blink_rate"] = df["AU45_c"].mean()
    elif "AU45_r" in df.columns:
        features["derived_blink_intensity_mean"] = df["AU45_r"].mean()

    # Gaze variability
    gaze_cols = [c for c in ["gaze_angle_x", "gaze_angle_y"] if c in df.columns]
    if len(gaze_cols) == 2:
        gaze_mag = np.sqrt(df["gaze_angle_x"]**2 + df["gaze_angle_y"]**2)
        features["derived_gaze_magnitude_mean"] = gaze_mag.mean()
        features["derived_gaze_magnitude_std"] = gaze_mag.std()

        gaze_shift = gaze_mag.diff().abs()
        features["derived_gaze_shift_mean"] = gaze_shift.mean()
        features["derived_gaze_shift_std"] = gaze_shift.std()

    # Head movement velocity
    pose_rot_cols = [c for c in ["pose_Rx", "pose_Ry", "pose_Rz"] if c in df.columns]
    if len(pose_rot_cols) == 3:
        rot_diff = df[pose_rot_cols].diff()
        head_vel = np.sqrt((rot_diff ** 2).sum(axis=1))
        features["derived_head_rotation_velocity_mean"] = head_vel.mean()
        features["derived_head_rotation_velocity_std"] = head_vel.std()

    pose_trans_cols = [c for c in ["pose_Tx", "pose_Ty", "pose_Tz"] if c in df.columns]
    if len(pose_trans_cols) == 3:
        trans_diff = df[pose_trans_cols].diff()
        trans_vel = np.sqrt((trans_diff ** 2).sum(axis=1))
        features["derived_head_translation_velocity_mean"] = trans_vel.mean()
        features["derived_head_translation_velocity_std"] = trans_vel.std()

    return features

In [4]:
def extract_features_for_participant(path, participant_id):
    raw_df = pd.read_csv(path)
    df = clean_openface_df(raw_df, confidence_threshold=0.6)

    features = {
        "Participant_ID": participant_id,
        "n_frames_raw": len(raw_df),
        "n_frames_clean": len(df),
        "frame_retention_rate": len(df) / max(len(raw_df), 1)
    }

    # Basic OpenFace groups
    au_cols = [c for c in df.columns if c.startswith("AU") and (c.endswith("_r") or c.endswith("_c"))]
    gaze_cols = [c for c in df.columns if c.startswith("gaze")]
    pose_cols = [c for c in df.columns if c.startswith("pose")]

    features.update(aggregate_columns(df, au_cols, "au"))
    features.update(aggregate_columns(df, gaze_cols, "gaze"))
    features.update(aggregate_columns(df, pose_cols, "pose"))
    features.update(extract_derived_video_features(df))

    return features

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive")

# Change this after checking your Drive structure
video_files_dest = BASE_DIR / "DAIC_WOZ_VIDEO_BRANCH" / "results" / "tables" / "video_file_availability.csv"

print("Exists:", video_files_dest.exists())
print("Path:", video_files_dest)

Exists: True
Path: /content/drive/MyDrive/DAIC_WOZ_VIDEO_BRANCH/results/tables/video_file_availability.csv


In [7]:
video_files_df = pd.read_csv(video_files_dest)
video_files_df.head()

,Participant_ID,filename,path
0,302,302_OpenFace2.1.0_Pose_gaze_AUs.csv,/content/drive/MyDrive/edaic/302/features/302_...
1,303,303_OpenFace2.1.0_Pose_gaze_AUs.csv,/content/drive/MyDrive/edaic/303/features/303_...
2,304,304_OpenFace2.1.0_Pose_gaze_AUs.csv,/content/drive/MyDrive/edaic/304/features/304_...
3,305,305_OpenFace2.1.0_Pose_gaze_AUs.csv,/content/drive/MyDrive/edaic/305/features/305_...
4,307,307_OpenFace2.1.0_Pose_gaze_AUs.csv,/content/drive/MyDrive/edaic/307/features/307_...


In [8]:
all_features = []

for _, row in video_files_df.iterrows():
    pid = row["Participant_ID"]
    path = row["path"]

    try:
        feats = extract_features_for_participant(path, pid)
        all_features.append(feats)
        print(f"Done: {pid}")
    except Exception as e:
        print(f"Failed: {pid} | {e}")

video_features_df = pd.DataFrame(all_features)
video_features_df.head()

Done: 302
Done: 303
Done: 304
Done: 305
Done: 307
Done: 308
Done: 309
Done: 310
Done: 311
Done: 312
Done: 313
Done: 314
Done: 315
Done: 316
Done: 318
Done: 319
Done: 322
Done: 323
Done: 324
Done: 325
Done: 326
Done: 327
Done: 328
Done: 329
Done: 330
Done: 332
Done: 333
Done: 335
Done: 337
Done: 338
Done: 339
Done: 340
Done: 341
Done: 345
Done: 346
Done: 348
Done: 349
Done: 351
Done: 352
Done: 353
Done: 354
Done: 355
Done: 356
Done: 357
Done: 358
Done: 359
Done: 360
Done: 361
Done: 362
Done: 363
Done: 364
Done: 366
Done: 367
Done: 368
Done: 369
Done: 370
Done: 372
Done: 375
Done: 376
Done: 377
Done: 378
Done: 379
Done: 380
Done: 383
Done: 384
Done: 385
Done: 386
Done: 387
Done: 389
Done: 390
Done: 391
Done: 392
Done: 395
Done: 396
Done: 397
Done: 399
Done: 400
Done: 403
Done: 404
Done: 405
Done: 406
Done: 407
Done: 409
Done: 410
Done: 411
Done: 413
Done: 414
Done: 416
Done: 417
Done: 418
Done: 419
Done: 420
Done: 421
Done: 422
Done: 424
Done: 426
Done: 427
Done: 428
Done: 429
Done: 430


,Participant_ID,n_frames_raw,n_frames_clean,frame_retention_rate,au_AU01_r_mean,au_AU01_r_std,au_AU01_r_min,au_AU01_r_max,au_AU01_r_median,au_AU01_r_p25,...,derived_au_expressiveness_std,derived_blink_rate,derived_gaze_magnitude_mean,derived_gaze_magnitude_std,derived_gaze_shift_mean,derived_gaze_shift_std,derived_head_rotation_velocity_mean,derived_head_rotation_velocity_std,derived_head_translation_velocity_mean,derived_head_translation_velocity_std
0,302,22766,22136,0.972327,0.253516,0.416087,0.0,3.26,0.02,0.0,...,0.192489,0.203695,0.289722,0.067919,0.015442,0.015481,0.010989,0.015854,2.051871,2.606517
1,303,29565,29400,0.994419,0.393095,0.638168,0.0,4.02,0.02,0.0,...,0.161447,0.296599,0.153982,0.088747,0.019067,0.024906,0.012901,0.011919,3.210802,3.545172
2,304,23780,23392,0.983684,0.271433,0.436940,0.0,3.25,0.02,0.0,...,0.156882,0.285397,0.300476,0.079532,0.016781,0.020224,0.013341,0.015578,2.667574,2.849901
3,305,51122,50975,0.997125,0.440564,0.673725,0.0,4.12,0.00,0.0,...,0.149619,0.332379,0.336748,0.075254,0.012763,0.013001,0.009825,0.017212,2.418475,2.906719
4,307,37167,36849,0.991444,0.408799,0.770625,0.0,5.00,0.02,0.0,...,0.167800,0.425222,0.229373,0.112786,0.014345,0.017032,0.017462,0.019672,2.849660,5.922183


In [9]:
PROCESSED_DIR = BASE_DIR / "DAIC_WOZ_VIDEO_BRANCH" / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

video_features_df.to_csv(PROCESSED_DIR / "video_session_features.csv", index=False)

In [10]:
video_features_df

,Participant_ID,n_frames_raw,n_frames_clean,frame_retention_rate,au_AU01_r_mean,au_AU01_r_std,au_AU01_r_min,au_AU01_r_max,au_AU01_r_median,au_AU01_r_p25,...,derived_au_expressiveness_std,derived_blink_rate,derived_gaze_magnitude_mean,derived_gaze_magnitude_std,derived_gaze_shift_mean,derived_gaze_shift_std,derived_head_rotation_velocity_mean,derived_head_rotation_velocity_std,derived_head_translation_velocity_mean,derived_head_translation_velocity_std
0,302,22766,22136,0.972327,0.253516,0.416087,0.0,3.26,0.02,0.0,...,0.192489,0.203695,0.289722,0.067919,0.015442,0.015481,0.010989,0.015854,2.051871,2.606517
1,303,29565,29400,0.994419,0.393095,0.638168,0.0,4.02,0.02,0.0,...,0.161447,0.296599,0.153982,0.088747,0.019067,0.024906,0.012901,0.011919,3.210802,3.545172
2,304,23780,23392,0.983684,0.271433,0.436940,0.0,3.25,0.02,0.0,...,0.156882,0.285397,0.300476,0.079532,0.016781,0.020224,0.013341,0.015578,2.667574,2.849901
3,305,51122,50975,0.997125,0.440564,0.673725,0.0,4.12,0.00,0.0,...,0.149619,0.332379,0.336748,0.075254,0.012763,0.013001,0.009825,0.017212,2.418475,2.906719
4,307,37167,36849,0.991444,0.408799,0.770625,0.0,5.00,0.02,0.0,...,0.167800,0.425222,0.229373,0.112786,0.014345,0.017032,0.017462,0.019672,2.849660,5.922183
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,695,45204,36080,0.798159,0.436593,0.665112,0.0,4.51,0.00,0.0,...,0.155904,0.329573,0.555334,0.123420,0.017452,0.022768,0.014070,0.026703,2.562187,17.203377
159,697,31754,31590,0.994835,0.821994,1.161346,0.0,4.77,0.00,0.0,...,0.138829,0.255239,0.518577,0.065852,0.016240,0.022406,0.009501,0.012601,1.930480,3.448503
160,702,21988,21862,0.994270,0.154501,0.292545,0.0,5.00,0.02,0.0,...,0.190293,0.302351,0.388517,0.042294,0.008080,0.010265,0.006555,0.012371,1.807959,2.928321
161,703,24391,24196,0.992005,0.441994,0.907870,0.0,5.00,0.01,0.0,...,0.176684,0.468094,0.439618,0.115421,0.019372,0.021400,0.011249,0.016332,2.133831,2.626695
